## 10. Walmart-ის გაყიდვების პროგნოზირება — Prophet v2 (Full Population + Regressors + LightGBM Correction)

| Field | Details |
|---|---|
| **მოდელი** | Prophet (per-series) + external regressors + LightGBM residual-correction stacking |
| **Logging** | MLflow / DagsHub · project: `walmart-forecasting` |
| **ექსპერიმენტი** | `Prophet_Training` |
| **მასშტაბი** | სრული Store/Dept პოპულაცია |
| **მიზანი** | validation WMAE 1100–1200 |

### რა შეიცვალა v1-თან შედარებით

- Prophet-ს ემატება დროში ცვალებადი გარეგანი რეგრესორები: `Temperature`, `Fuel_Price`, `CPI`, `Unemployment`, `MarkDown_sum` (features.csv-დან, store-level, ffill/bfill + median impute).
- Thanksgiving/Christmas holiday window გაფართოებულია (`-2`/`+2`), holidays_prior_scale და yearly Fourier order-ის ძებნის დიაპაზონი გაზრდილია, დამატებულია `regressor_prior_scale`.
- Prophet-ის თითოეული series-ის ნარჩენებზე (train-period residuals: `y - yhat`) გაწვრთნილია LightGBM correction მოდელი Store/Dept/Type/Size/თვე/კვირა/holiday/რეგრესორების feature-ებზე — ეს არის ის cross-series ინფორმაცია, რომელსაც ცალკეული Prophet fit-ები ვერ ხედავენ.
- საბოლოო რიცხვი არის **Prophet + LightGBM correction**-ის val WMAE, არა წმინდა Prophet-ის.

**გამჭვირვალობისთვის:** საჯარო ბენჩმარკებით (იხ. summary-ის ცხრილი) სუფთა Prophet ამ კონკურსზე ჩვეულებრივ ~2,500–3,500-ის დიაპაზონშია, ხოლო feature-engineered GBM-ონლი გადაწყვეტები ~1,300–1,450-ზეა. 1,100–1,200 ამ metric-ისთვის ძალიან აგრესიული სამიზნეა — ეს notebook იყენებს ყველა ხელმისაწვდომ ტექნიკას ამ დიაპაზონისკენ მისაახლოებლად, მაგრამ ზუსტი რიცხვი დამოკიდებულია რეალურ მონაცემებზე და უნდა შემოწმდეს გაშვებით.

## 1. Setup

In [ ]:
!pip install prophet mlflow dagshub joblib lightgbm scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from prophet import Prophet
import mlflow
import lightgbm as lgb

import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

import time
import itertools
import random
from joblib import Parallel, delayed

print("Prophet, LightGBM, MLflow ready")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
MODELS_DIR = f'{PROJECT_DIR}/models'

import os
os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
import dagshub

DAGSHUB_USERNAME = "zberi23"
DAGSHUB_REPO = "walmart-forecasting"

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

EXPERIMENT_NAME = "Prophet_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 2. მონაცემების ჩატვირთვა

In [ ]:
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
features_raw = pd.read_csv(f'{DATA_DIR}/features.csv.zip')
stores_raw = pd.read_csv(f'{DATA_DIR}/stores.csv')

train_raw['Date'] = pd.to_datetime(train_raw['Date'])
test_raw['Date'] = pd.to_datetime(test_raw['Date'])
features_raw['Date'] = pd.to_datetime(features_raw['Date'])

print(f"Train: {train_raw.shape}")
print(f"Test:  {test_raw.shape}")
print(f"Features: {features_raw.shape}")
print(f"Stores: {stores_raw.shape}")

## 3. Feature engineering — რეგრესორები და holiday calendar

In [ ]:
markdown_cols = [c for c in features_raw.columns if c.startswith('MarkDown')]
feat = features_raw.copy()
feat[markdown_cols] = feat[markdown_cols].fillna(0.0)
feat['MarkDown_sum'] = feat[markdown_cols].sum(axis=1)

feat = feat.sort_values(['Store', 'Date'])
for col in ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
    feat[col] = feat.groupby('Store')[col].transform(lambda s: s.ffill().bfill())
    feat[col] = feat[col].fillna(feat[col].median())

REGRESSOR_COLS = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'MarkDown_sum']
store_feat = feat[['Store', 'Date'] + REGRESSOR_COLS].drop_duplicates(subset=['Store', 'Date']).reset_index(drop=True)
store_feat_by_store = {s: g.set_index('Date').sort_index() for s, g in store_feat.groupby('Store')}
store_meta = stores_raw.set_index('Store')


def attach_regressors(g):
    return g.merge(store_feat, on=['Store', 'Date'], how='left')


def get_future_regressors(store, ds_array):
    sdf = store_feat_by_store.get(store)
    if sdf is None:
        return pd.DataFrame({c: np.zeros(len(ds_array)) for c in REGRESSOR_COLS})
    out = sdf.reindex(pd.DatetimeIndex(ds_array))[REGRESSOR_COLS]
    med = sdf[REGRESSOR_COLS].median()
    out = out.fillna(med)
    return out.reset_index(drop=True)


print("Regressor columns:", REGRESSOR_COLS)
store_feat.head()

In [ ]:
holidays_df = pd.DataFrame({
    'holiday': (['SuperBowl'] * 4 + ['LaborDay'] * 4 +
                ['Thanksgiving'] * 4 + ['Christmas'] * 4),
    'ds': pd.to_datetime([
        '2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08',
        '2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06',
        '2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29',
        '2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27',
    ]),
    'lower_window': [-1, -1, -1, -1, -1, -1, -1, -1, -2, -2, -2, -2, -2, -2, -2, -2],
    'upper_window': [1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2],
})

HOLIDAY_DATES = set(pd.to_datetime(holidays_df['ds']).to_numpy())
print(holidays_df['holiday'].value_counts())

## 4. Train/Validation split + WMAE

In [ ]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(np.asarray(is_holiday), 5.0, 1.0)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


VAL_WEEKS = 12

series_frames = {}
val_frames = {}

for (store, dept), g in train_raw.groupby(['Store', 'Dept'], sort=False):
    g = g.sort_values('Date')
    if len(g) <= VAL_WEEKS + 10:
        continue
    g = attach_regressors(g)
    for col in REGRESSOR_COLS:
        g[col] = g[col].fillna(g[col].median() if g[col].notna().any() else 0.0)
    tr = g.iloc[:-VAL_WEEKS]
    va = g.iloc[-VAL_WEEKS:]
    series_frames[(store, dept)] = tr.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})[['ds', 'y'] + REGRESSOR_COLS].reset_index(drop=True)
    val_frames[(store, dept)] = va.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})[['ds', 'y', 'IsHoliday'] + REGRESSOR_COLS].reset_index(drop=True)

print(f"Series eligible for train/val split: {len(series_frames)}")
print(f"Series dropped (too short): {train_raw.groupby(['Store', 'Dept']).ngroups - len(series_frames)}")

overlap_check = set(series_frames[list(series_frames.keys())[0]]['ds']) & set(val_frames[list(series_frames.keys())[0]]['ds'])
print("Sanity check — train/val date overlap for first series (must be empty):", overlap_check)

## 5. Prophet model builder + fit harness

In [ ]:
def build_model(cfg):
    model = Prophet(
        holidays=holidays_df,
        yearly_seasonality=cfg['yearly_order'],
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode=cfg['seasonality_mode'],
        changepoint_prior_scale=cfg['changepoint_prior_scale'],
        holidays_prior_scale=cfg['holidays_prior_scale'],
    )
    for col in REGRESSOR_COLS:
        model.add_regressor(col, prior_scale=cfg['regressor_prior_scale'], mode=cfg['seasonality_mode'])
    return model


def fit_one_series(key, frame, future_frame, cfg):
    import logging
    logging.getLogger('cmdstanpy').setLevel(logging.ERROR)
    logging.getLogger('prophet').setLevel(logging.WARNING)

    model = build_model(cfg)
    model.fit(frame[['ds', 'y'] + REGRESSOR_COLS])

    forecast = model.predict(future_frame[['ds'] + REGRESSOR_COLS])
    in_sample = model.predict(frame[['ds'] + REGRESSOR_COLS])

    return {
        'key': key,
        'val_pred': forecast['yhat'].to_numpy(),
        'val_ds': future_frame['ds'].to_numpy(),
        'val_regs': future_frame[REGRESSOR_COLS].to_numpy(),
        'train_true': frame['y'].to_numpy(),
        'train_pred': in_sample['yhat'].to_numpy(),
        'train_ds': frame['ds'].to_numpy(),
        'train_regs': frame[REGRESSOR_COLS].to_numpy(),
    }


def naive_forecast(key, horizon):
    y_hist = series_frames[key]['y'].values
    if len(y_hist) >= 52:
        return np.full(horizon, y_hist[-52])
    return np.full(horizon, y_hist[-1])

## 6. ფიქსირებული შეფასების sample + ჰიპერპარამეტრების ძებნის harness

Prophet-ის თითოეული trial ცალკეულ series-ზეა fit, ამიტომ 20+ კონფიგურაციის სრულ პოპულაციაზე გაშვება არარეალურია დროში.
ყველა trial ეშვება ერთსა და იმავე, ერთხელ შემთხვევით შერჩეულ series-ების ფიქსირებულ ქვესიმრავლეზე.

In [ ]:
SAMPLE_SIZE = 250
eligible_keys = [k for k, v in series_frames.items() if len(v) >= 60]

rng = np.random.default_rng(0)
sample_idx = rng.permutation(len(eligible_keys))[:SAMPLE_SIZE]
EVAL_SAMPLE = [eligible_keys[i] for i in sample_idx]

print(f"Tuning on a fixed sample of {len(EVAL_SAMPLE)} series")

In [ ]:
def run_trial(cfg, keys=None, run_name=None, log_to_mlflow=True):
    keys = EVAL_SAMPLE if keys is None else keys

    fit_keys = [k for k in keys if k in series_frames and len(series_frames[k]) >= cfg['min_history']]
    naive_keys = [k for k in keys if k not in fit_keys]

    t0 = time.time()
    results = Parallel(n_jobs=cfg['n_jobs'])(
        delayed(fit_one_series)(
            k, series_frames[k], val_frames[k][['ds'] + REGRESSOR_COLS], cfg
        ) for k in fit_keys
    )
    elapsed = time.time() - t0

    val_true, val_pred, val_hol = [], [], []
    train_true, train_pred, train_hol = [], [], []

    for r in results:
        key = r['key']
        va = val_frames[key]
        val_true.append(va['y'].values)
        val_pred.append(r['val_pred'])
        val_hol.append(va['IsHoliday'].values)

        train_true.append(r['train_true'])
        train_pred.append(r['train_pred'])
        train_hol.append(np.isin(r['train_ds'], list(HOLIDAY_DATES)))

    for k in naive_keys:
        va = val_frames[k]
        val_true.append(va['y'].values)
        val_pred.append(naive_forecast(k, len(va)))
        val_hol.append(va['IsHoliday'].values)

    val_true = np.concatenate(val_true)
    val_pred = np.concatenate(val_pred)
    val_hol = np.concatenate(val_hol)
    val_wmae = wmae(val_true, val_pred, val_hol)
    val_mae = np.mean(np.abs(val_true - val_pred))
    val_rmse = float(np.sqrt(np.mean((val_true - val_pred) ** 2)))
    ss_res = np.sum((val_true - val_pred) ** 2)
    ss_tot = np.sum((val_true - val_true.mean()) ** 2)
    val_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    if train_true:
        train_true_arr = np.concatenate(train_true)
        train_pred_arr = np.concatenate(train_pred)
        train_hol_arr = np.concatenate(train_hol)
        train_wmae = wmae(train_true_arr, train_pred_arr, train_hol_arr)
        train_mae = np.mean(np.abs(train_true_arr - train_pred_arr))
    else:
        train_wmae = float('nan')
        train_mae = float('nan')

    metrics = {
        'val_wmae': val_wmae, 'val_mae': val_mae, 'val_rmse': val_rmse, 'val_r2': val_r2,
        'train_wmae': train_wmae, 'train_mae': train_mae,
        'n_fit': len(fit_keys), 'n_fallback': len(naive_keys), 'elapsed_sec': elapsed,
    }

    if log_to_mlflow:
        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({k: v for k, v in cfg.items() if k != 'n_jobs'})
            mlflow.log_metrics(metrics)
            mlflow.set_tag('stage', 'hyperparameter_search')

    return metrics, results

## 7. ჰიპერპარამეტრების grid (24 კონფიგურაცია) და ძებნის გაშვება

In [ ]:
N_JOBS = -1

BASE_CFG = dict(
    seasonality_mode='multiplicative', changepoint_prior_scale=0.05,
    holidays_prior_scale=30.0, yearly_order=15, regressor_prior_scale=1.0,
    min_history=100, n_jobs=N_JOBS,
)

grid = list(itertools.product(
    ['additive', 'multiplicative'],
    [0.01, 0.05, 0.1, 0.3, 0.5],
    [10.0, 20.0, 30.0, 50.0],
    [10, 15, 20],
    [0.1, 1.0, 5.0, 10.0],
    [60, 100, 140],
))

base_tuple = (BASE_CFG['seasonality_mode'], BASE_CFG['changepoint_prior_scale'],
              BASE_CFG['holidays_prior_scale'], BASE_CFG['yearly_order'],
              BASE_CFG['regressor_prior_scale'], BASE_CFG['min_history'])

rng_grid = random.Random(42)
rng_grid.shuffle(grid)
sampled = [g for g in grid if g != base_tuple][:23]

SEARCH_CONFIGS = [BASE_CFG] + [
    dict(seasonality_mode=m, changepoint_prior_scale=cps, holidays_prior_scale=hps,
         yearly_order=yo, regressor_prior_scale=rps, min_history=mh, n_jobs=N_JOBS)
    for (m, cps, hps, yo, rps, mh) in sampled
]

print(f"Total configurations to trial: {len(SEARCH_CONFIGS)}")
SEARCH_CONFIGS[:3]

In [ ]:
search_log = []

for i, cfg in enumerate(SEARCH_CONFIGS):
    run_name = f"Prophet_Search_{i:02d}"
    metrics, _ = run_trial(cfg, run_name=run_name)
    row = {**{k: v for k, v in cfg.items() if k != 'n_jobs'}, **metrics, 'trial': i, 'run_name': run_name}
    search_log.append(row)
    print(f"[{i:02d}] mode={cfg['seasonality_mode']:<14} cps={cfg['changepoint_prior_scale']:<5} "
          f"hps={cfg['holidays_prior_scale']:<5} yo={cfg['yearly_order']:<3} rps={cfg['regressor_prior_scale']:<5} "
          f"min_hist={cfg['min_history']:<4} -> val_wmae={metrics['val_wmae']:.2f}")

search_df = pd.DataFrame(search_log)

## 8. საუკეთესო კონფიგურაციის შერჩევა

In [ ]:
search_df_sorted = search_df.sort_values('val_wmae').reset_index(drop=True)
display(search_df_sorted.head(10))

BEST_ROW = search_df_sorted.iloc[0]
FINAL_CFG = dict(
    seasonality_mode=BEST_ROW['seasonality_mode'],
    changepoint_prior_scale=BEST_ROW['changepoint_prior_scale'],
    holidays_prior_scale=BEST_ROW['holidays_prior_scale'],
    yearly_order=int(BEST_ROW['yearly_order']),
    regressor_prior_scale=BEST_ROW['regressor_prior_scale'],
    min_history=int(BEST_ROW['min_history']),
    n_jobs=N_JOBS,
)
print("Best config (on fixed sample):", FINAL_CFG)
print("Sample val_wmae:", BEST_ROW['val_wmae'])

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(search_df_sorted))
width = 0.35
ax.bar(x - width / 2, search_df_sorted['train_wmae'], width, label='train_wmae', color='#9fc5e8')
ax.bar(x + width / 2, search_df_sorted['val_wmae'], width, label='val_wmae', color='#e06666')
ax.axvspan(-0.5, 0.5, color='gold', alpha=0.2)
ax.set_xticks(x)
ax.set_xticklabels(search_df_sorted['trial'], rotation=0)
ax.set_xlabel('Trial (sorted by val_wmae)')
ax.set_ylabel('WMAE ($)')
ax.set_title('Prophet hyperparameter search — train vs val WMAE per trial')
ax.legend()
plt.tight_layout()
plt.show()

## 9. სრული პოპულაციის შეფასება საუკეთესო კონფიგურაციით (Prophet-only)

In [ ]:
ALL_KEYS = list(series_frames.keys())
print(f"Full population size: {len(ALL_KEYS)} series")

full_metrics, full_results = run_trial(FINAL_CFG, keys=ALL_KEYS, run_name="Prophet_Final", log_to_mlflow=False)
full_pred_map = {r['key']: r['val_pred'] for r in full_results}

with mlflow.start_run(run_name="Prophet_Final") as final_run:
    mlflow.log_params({k: v for k, v in FINAL_CFG.items() if k != 'n_jobs'})
    mlflow.log_param('sample_size', SAMPLE_SIZE)
    mlflow.log_param('n_series_full', len(ALL_KEYS))
    mlflow.log_metrics(full_metrics)
    mlflow.set_tag('stage', 'full_population')
    mlflow.set_tag('model_type', 'Prophet')

print("Prophet-only FULL population val WMAE:", full_metrics['val_wmae'])
print("Fitted series:", full_metrics['n_fit'], "| Naive fallback series:", full_metrics['n_fallback'])

## 9.5 LightGBM residual-correction stacking

Prophet-ის თითოეული series ცალკეა fit, ამიტომ ვერ სწავლობს Store/Dept/Type/Size-ს შორის საერთო paternebs.
აქ ვწვრთნით LightGBM-ს Prophet-ის train-period ნარჩენებზე (`y - yhat`) და ვამატებთ correction-ს val prediction-ს.

In [ ]:
CORR_FEATURES = ['Store', 'Dept', 'Type', 'Size', 'Month', 'WeekOfYear', 'IsHoliday'] + REGRESSOR_COLS


def make_correction_table(results, holiday_dates):
    rows = []
    targets = []
    for r in results:
        store, dept = r['key']
        meta = store_meta.loc[store]
        dates = pd.DatetimeIndex(r['train_ds'])
        df = pd.DataFrame({
            'Store': store, 'Dept': dept,
            'Type': meta['Type'], 'Size': meta['Size'],
            'Month': dates.month, 'WeekOfYear': dates.isocalendar().week.astype(int),
            'IsHoliday': np.isin(r['train_ds'], list(holiday_dates)).astype(int),
        })
        for i, col in enumerate(REGRESSOR_COLS):
            df[col] = r['train_regs'][:, i]
        rows.append(df)
        targets.append(r['train_true'] - r['train_pred'])
    X = pd.concat(rows, ignore_index=True)
    y = np.concatenate(targets)
    return X, y


def build_val_eval_table(results):
    true_list, prophet_list, hol_list = [], [], []
    feat_rows = []
    for r in results:
        store, dept = r['key']
        meta = store_meta.loc[store]
        va = val_frames[r['key']]
        true_list.append(va['y'].to_numpy())
        prophet_list.append(r['val_pred'])
        hol_list.append(va['IsHoliday'].to_numpy())
        dates = pd.DatetimeIndex(r['val_ds'])
        f = pd.DataFrame({
            'Store': store, 'Dept': dept, 'Type': meta['Type'], 'Size': meta['Size'],
            'Month': dates.month, 'WeekOfYear': dates.isocalendar().week.astype(int),
            'IsHoliday': va['IsHoliday'].to_numpy().astype(int),
        })
        for i, col in enumerate(REGRESSOR_COLS):
            f[col] = r['val_regs'][:, i]
        feat_rows.append(f)
    return (np.concatenate(true_list), np.concatenate(prophet_list),
            np.concatenate(hol_list).astype(bool), pd.concat(feat_rows, ignore_index=True))


X_train_corr, y_train_corr = make_correction_table(full_results, HOLIDAY_DATES)
X_train_corr['Type'] = X_train_corr['Type'].astype('category')

corr_model = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=42,
)
corr_model.fit(X_train_corr[CORR_FEATURES], y_train_corr, categorical_feature=['Type'])
print("Correction model trained on", len(X_train_corr), "rows")

In [ ]:
val_true_arr, val_pred_prophet_arr, val_hol_arr, val_feat_df = build_val_eval_table(full_results)

val_feat_df['Type'] = pd.Categorical(val_feat_df['Type'], categories=X_train_corr['Type'].cat.categories)
val_correction = corr_model.predict(val_feat_df[CORR_FEATURES])
val_pred_corrected_fit = val_pred_prophet_arr + val_correction

naive_keys = [k for k in ALL_KEYS if k not in full_pred_map]
naive_true, naive_pred, naive_hol = [], [], []
for k in naive_keys:
    va = val_frames[k]
    naive_true.append(va['y'].to_numpy())
    naive_pred.append(naive_forecast(k, len(va)))
    naive_hol.append(va['IsHoliday'].to_numpy())

if naive_true:
    naive_true_arr = np.concatenate(naive_true)
    naive_pred_arr = np.concatenate(naive_pred)
    naive_hol_arr = np.concatenate(naive_hol).astype(bool)
else:
    naive_true_arr = np.array([])
    naive_pred_arr = np.array([])
    naive_hol_arr = np.array([], dtype=bool)

val_wmae_corrected = wmae(
    np.concatenate([val_true_arr, naive_true_arr]),
    np.concatenate([val_pred_corrected_fit, naive_pred_arr]),
    np.concatenate([val_hol_arr, naive_hol_arr]),
)

print("Prophet-only val WMAE:", full_metrics['val_wmae'])
print("Prophet + LightGBM correction val WMAE:", val_wmae_corrected)

with mlflow.start_run(run_name="Prophet_Corrected_Final") as corrected_run:
    mlflow.log_params({k: v for k, v in FINAL_CFG.items() if k != 'n_jobs'})
    mlflow.log_metric('val_wmae_prophet_only', full_metrics['val_wmae'])
    mlflow.log_metric('val_wmae_corrected', val_wmae_corrected)
    mlflow.log_param('correction_model', 'lightgbm')
    mlflow.set_tag('stage', 'residual_correction')
    mlflow.set_tag('model_type', 'Prophet+LightGBM')

print(f"Target range: 1100-1200 | Achieved corrected val WMAE: {val_wmae_corrected:.2f}")

## 10. Actual vs Predicted (Prophet + correction)

In [ ]:
plot_true = np.concatenate([val_true_arr, naive_true_arr])
plot_pred = np.concatenate([val_pred_corrected_fit, naive_pred_arr])
plot_hol = np.concatenate([val_hol_arr, naive_hol_arr])

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(plot_true[~plot_hol], plot_pred[~plot_hol], s=6, alpha=0.3, color='#6fa8dc', label='non-holiday')
ax.scatter(plot_true[plot_hol], plot_pred[plot_hol], s=12, alpha=0.6, color='#e06666', label='holiday (5x weight)')
lo = min(plot_true.min(), plot_pred.min())
hi = max(plot_true.max(), plot_pred.max())
ax.plot([lo, hi], [lo, hi], color='black', linestyle='--', linewidth=1, label='y = ŷ')
ax.set_xlabel('Actual Weekly_Sales')
ax.set_ylabel('Predicted Weekly_Sales')
ax.set_title(f"Prophet + LightGBM correction — val_wmae={val_wmae_corrected:.2f}")
ax.legend()
plt.tight_layout()
plt.show()

## 11. საბოლოო refit + test პროგნოზი

In [ ]:
TEST_DATES = np.sort(test_raw['Date'].unique())
print(f"Test horizon: {len(TEST_DATES)} weeks, {TEST_DATES[0]} -> {TEST_DATES[-1]}")

full_train_frames = {}
for (store, dept), g in train_raw.groupby(['Store', 'Dept'], sort=False):
    g = g.sort_values('Date')
    g = attach_regressors(g)
    for col in REGRESSOR_COLS:
        g[col] = g[col].fillna(g[col].median() if g[col].notna().any() else 0.0)
    full_train_frames[(store, dept)] = g.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})[['ds', 'y'] + REGRESSOR_COLS].reset_index(drop=True)

refit_keys = [k for k, v in full_train_frames.items() if len(v) >= FINAL_CFG['min_history']]
print(f"Series refit on full train history: {len(refit_keys)} / {len(full_train_frames)}")


def make_test_future(key):
    store, dept = key
    regs = get_future_regressors(store, TEST_DATES)
    fdf = pd.DataFrame({'ds': TEST_DATES})
    for col in REGRESSOR_COLS:
        fdf[col] = regs[col].to_numpy()
    return fdf


refit_results = Parallel(n_jobs=N_JOBS)(
    delayed(fit_one_series)(k, full_train_frames[k], make_test_future(k), FINAL_CFG) for k in refit_keys
)
refit_pred_map = {r['key']: r['val_pred'] for r in refit_results}

In [ ]:
X_train_corr2, y_train_corr2 = make_correction_table(refit_results, HOLIDAY_DATES)
X_train_corr2['Type'] = X_train_corr2['Type'].astype('category')

corr_model_final = lgb.LGBMRegressor(
    n_estimators=600, learning_rate=0.03, num_leaves=63,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=42,
)
corr_model_final.fit(X_train_corr2[CORR_FEATURES], y_train_corr2, categorical_feature=['Type'])
print("Final correction model trained on", len(X_train_corr2), "rows")

In [ ]:
def naive_test_forecast(key, horizon):
    y_hist = full_train_frames[key]['y'].values
    if len(y_hist) >= 52:
        return np.full(horizon, y_hist[-52])
    return np.full(horizon, y_hist[-1])


def build_test_feat(key, ds, regs):
    store, dept = key
    meta = store_meta.loc[store]
    dates = pd.DatetimeIndex(ds)
    f = pd.DataFrame({
        'Store': store, 'Dept': dept, 'Type': meta['Type'], 'Size': meta['Size'],
        'Month': dates.month, 'WeekOfYear': dates.isocalendar().week.astype(int),
        'IsHoliday': np.isin(ds, list(HOLIDAY_DATES)).astype(int),
    })
    for i, col in enumerate(REGRESSOR_COLS):
        f[col] = regs[:, i]
    return f


refit_by_key = {r['key']: r for r in refit_results}

submission_rows = []
for (store, dept), g in test_raw.groupby(['Store', 'Dept'], sort=False):
    g = g.sort_values('Date').reset_index(drop=True)
    key = (store, dept)
    if key in refit_by_key:
        r = refit_by_key[key]
        idx = np.isin(TEST_DATES, g['Date'].to_numpy())
        base_pred = r['val_pred'][idx][:len(g)]
        feat = build_test_feat(key, r['val_ds'][idx][:len(g)], r['val_regs'][idx][:len(g)])
        feat['Type'] = pd.Categorical(feat['Type'], categories=X_train_corr2['Type'].cat.categories)
        corr = corr_model_final.predict(feat[CORR_FEATURES])
        preds = base_pred + corr
    elif key in full_train_frames:
        preds = naive_test_forecast(key, len(g))
    else:
        preds = np.full(len(g), train_raw['Weekly_Sales'].median())

    ids = g['Store'].astype(str) + '_' + g['Dept'].astype(str) + '_' + g['Date'].dt.strftime('%Y-%m-%d')
    submission_rows.append(pd.DataFrame({'Id': ids, 'Weekly_Sales': preds}))

submission = pd.concat(submission_rows, ignore_index=True)
print(submission.shape)
submission.head()

In [ ]:
submission_path = f'{MODELS_DIR}/prophet_v2_submission.csv'
submission.to_csv(submission_path, index=False)
search_df.to_csv(f'{MODELS_DIR}/prophet_v2_search_log.csv', index=False)
print(f"Saved submission: {submission_path}")

## 12. Prophet Component Decomposition (მაგალითი)

In [ ]:
example_key = max(full_train_frames, key=lambda k: len(full_train_frames[k]))
example_frame = full_train_frames[example_key]

example_model = build_model(FINAL_CFG)
example_model.fit(example_frame[['ds', 'y'] + REGRESSOR_COLS])

future_ds = pd.date_range(example_frame['ds'].max(), periods=40, freq='W-FRI')[1:]
future_regs = get_future_regressors(example_key[0], future_ds)
future_df = pd.DataFrame({'ds': future_ds})
for col in REGRESSOR_COLS:
    future_df[col] = future_regs[col].to_numpy()

example_forecast = example_model.predict(pd.concat([example_frame[['ds'] + REGRESSOR_COLS], future_df], ignore_index=True))

fig = example_model.plot_components(example_forecast)
plt.suptitle(f'Prophet Components: Store{example_key[0]}_Dept{example_key[1]}', y=1.02)
plt.tight_layout()
plt.show()

## 13. მოდელების და შედეგების შენახვა

In [ ]:
import pickle

artifact_path = f'{MODELS_DIR}/prophet_v2_full_population_results.pkl'
with open(artifact_path, 'wb') as f:
    pickle.dump({
        'final_cfg': FINAL_CFG,
        'search_log': search_df,
        'full_metrics': full_metrics,
        'val_wmae_corrected': val_wmae_corrected,
        'holidays_df': holidays_df,
        'corr_model_final': corr_model_final,
        'example_model': example_model,
        'example_key': example_key,
        'n_series_total': len(full_train_frames),
        'n_series_refit': len(refit_keys),
    }, f)

print(f"Saved: {artifact_path}")
print(f"Full population val WMAE (Prophet only): {full_metrics['val_wmae']:.2f}")
print(f"Full population val WMAE (Prophet + correction): {val_wmae_corrected:.2f}")

## 14. შეჯამება

| ეტაპი | აღწერა |
|---|---|
| **Hyperparameter search** | 24 კონფიგურაცია (`seasonality_mode`, `changepoint_prior_scale`, `holidays_prior_scale`, yearly Fourier order, `regressor_prior_scale`, `min_history`) ფიქსირებულ 250-series sample-ზე |
| **Regressors** | Temperature, Fuel_Price, CPI, Unemployment, MarkDown_sum — დროში ცვალებადი, Prophet `add_regressor` |
| **Full population run** | საუკეთესო კონფიგურაცია მთელ Store/Dept პოპულაციაზე |
| **Residual correction** | LightGBM ნარჩენებზე (Store/Dept/Type/Size/თვე/კვირა/holiday/რეგრესორები) |
| **Headline metric** | `val_wmae_corrected` (Prophet + LightGBM), ინდივიდუალურ Prophet-ონლი მაჩვენებელზე დაბალი |
| **Submission** | refit მთელ history-ზე + საბოლოო correction model, 39-კვირიანი test პროგნოზი |

### გამჭვირვალობა სამიზნესთან მიმართებით

წყარო/ID | სავარაუდო WMAE (იმავე last-12-week holdout მეთოდოლოგიით)
---|---
სუფთა Prophet (ეს notebook, v1) | ~2,500–3,500 დიაპაზონი საჯარო post-ების მიხედვით
feature-engineered RF/XGBoost (საჯარო) | ~1,300–1,450
**ეს notebook (v2): Prophet + regressors + LightGBM correction** | იხილეთ ზემოთ ნაბეჭდი `val_wmae_corrected`
სამიზნე | 1,100–1,200

1,100–1,200-ის მიღწევა ამ metric-ისთვის საჯარო ბენჩმარკებზე დაბალია, ვიდრე თვით feature-engineered GBM-ონლი გადაწყვეტები — ეს ნიშნავს, რომ ეს არის ძალიან აგრესიული სამიზნე. ეს notebook იყენებს ყველა რეალისტურ ტექნიკას (გარეგანი რეგრესორები, გაფართოებული holiday window, LightGBM stacking) მაქსიმალურად დასაახლოებლად, მაგრამ ზუსტი შედეგი დამოკიდებულია რეალურ მონაცემებზე და უნდა შემოწმდეს რეალური გაშვებით.